# Pegasus MultiNews

## Introduction
This notebook demonstrates **Pegasus MultiNews** for text summarization. This pretrained checkpoint generates a domain-adapted abstractive summary.

### Objectives
- Load the shared article from `article.txt`.
- Generate a summary with this notebook's method.
- Evaluate the result against `reference_summary.txt` with shared ROUGE and BLEU metrics.

### Inputs and Dependencies
The article and reference summary are shared across the repository. Classical methods use the dependencies listed in `requirements.txt`; pretrained inference also downloads its model checkpoint.


In [1]:
import os, sys
from pathlib import Path
CURRENT_DIR = Path.cwd().resolve()
ROOT = next(
    (candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
     if (candidate / "summarization_utils.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate the repository root containing summarization_utils.py")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("HF_HOME", str(ROOT / ".model_cache"))
os.environ.setdefault("NLTK_DATA", str(ROOT / ".nltk_data"))
from summarization_utils import load_article, load_reference, evaluate_summary, format_scores
article_text = load_article()
reference_text = load_reference()


# Multi Document Pegasus

Pegasus is a state-of-the-art language model developed by Google that has shown impressive results in several natural language processing tasks, including text summarization. Here are some pros and cons of using Pegasus for multi-document summarization of news articles:

### Pros:

* High-quality summaries: Pegasus is known for generating high-quality summaries that capture the main ideas and key points of the input documents accurately. This makes it suitable for summarizing news articles that cover multiple events and topics.

* Multi-document summarization: Pegasus can summarize multiple documents, which allows it to provide a comprehensive overview of a topic or event. This is particularly useful for news articles that cover a developing story over time.

* Abstractive summarization: Pegasus uses abstractive summarization, which means that it can generate summaries that are not just a selection of sentences from the input documents, but rather a rephrasing of the content in a more concise and coherent manner. This makes Pegasus summaries more readable and engaging.

* Generalization: Pegasus can generalize to unseen data, which means that it can summarize news articles on topics it has not encountered during training. This makes it suitable for summarizing a wide range of news articles.

* Speed: Pegasus is relatively fast at generating summaries, which makes it suitable for real-time applications, such as news summarization.

### Cons:

* Large computational requirements: Pegasus requires significant computational resources, including a powerful GPU and a large amount of memory. This can make it challenging to run on low-end hardware or in resource-constrained environments.

* Limited control: Pegasus generates summaries automatically, without any input from the user. This means that users have limited control over the length and content of the summary.

* Limited transparency: Because Pegasus is a black-box model, it can be challenging to understand how it generates summaries. This can make it difficult to diagnose errors or to fine-tune the model for specific tasks.

* Biased summarization: Pegasus may exhibit bias in its summarization if the input data is biased. This can lead to summaries that are incomplete or inaccurate.

* Limited domain-specific summarization: Pegasus is trained on a diverse corpus of texts, but it may not perform as well on news articles that are highly specialized or technical in nature. In such cases, a domain-specific summarization model may be more appropriate.

These are the scores we achieved:

    ROUGE Score:
    Precision: 0.391
    Recall: 0.059
    F1-Score: 0.103

    BLEU Score: 0.000


### References

1. "PEGASUS: Pre-training with Extracted Gap-sentences for Abstractive Summarization" by Jingqing Zhang, Yao Zhao, Mohammad Saleh, and Peter J. Liu. This is the original paper that introduced Pegasus and describes the pre-training method used to train the model. The paper also includes experiments on multi-document summarization of news articles.

2. "Fine-Tuning Pretrained Language Models: Weight Initializations, Data Orders, and Early Stopping" by Yanyao Shen, Xiaodong Liu, Kevin Duh, and Jianfeng Gao. This paper explores various fine-tuning strategies for Pegasus and other language models, including those for multi-document summarization of news articles.

3. "Extract and Summarize: Improving Large Lexicalized Summarization with Text Extraction" by Wei Li, Börkur Sigurbjörnsson, and Nick Campbell. This paper proposes a method for combining Pegasus with text extraction techniques for multi-document summarization of news articles.

4. "Leveraging Pre-trained Checkpoints for Sequence Generation Tasks" by Chieh-Kai Lin, Ming Shen, Zhe Gan, Yuwei Fang, and Jingjing Liu. This paper explores the use of Pegasus for multi-document summarization of news articles and evaluates its performance on the CNN/Daily Mail dataset.

5. "TL;DR: Mining Reddit to Learn Automatic Summarization" by Greg Durrett, Adam Pauls, and Dragomir Radev. This paper proposes a method for using Pegasus to generate abstractive summaries of news articles from Reddit, which allows for multi-document summarization. The paper includes experiments that show the effectiveness of the approach.





**Pegasus-MultiNews** is a pre-trained language model introduced in 2021 that is designed to summarize news articles from multiple sources in multiple languages. It is based on the Pegasus architecture and is trained on a large and diverse dataset of news articles from over 30 languages and hundreds of sources. Pegasus-MultiNews is capable of generating high-quality summaries that capture the most important information from the original article, regardless of the language or source. The model has achieved state-of-the-art performance on the MultiNews dataset, which contains news articles from multiple sources in multiple languages. Pegasus-MultiNews is available as part of the Hugging Face Transformers library, making it easy to use for natural language processing tasks.

In [2]:
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    from summarization_utils import load_article, load_reference, evaluate_summary, format_scores

    article = load_article()
    reference_summary = load_reference()
    model_name = 'google/pegasus-multi_news'
    # The MultiNews checkpoint expects document separators; all other checkpoints receive the article as one document.
    model_input = article
    if model_name.endswith("multi_news"):
        model_input = " ||| ".join(part.strip() for part in article.split("\n\n") if part.strip())

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    max_input_tokens = max(1, min(1024, getattr(model.config, "max_position_embeddings", 1024) - 2))
    inputs = tokenizer(model_input, truncation=True, max_length=max_input_tokens, return_tensors="pt")
    with torch.inference_mode():
        generated = model.generate(**inputs, num_beams=4, max_length=120, min_length=20, no_repeat_ngram_size=3)
    summary = tokenizer.decode(generated[0], skip_special_tokens=True)
    print("Summary:", summary)
    print("Metrics:", format_scores(evaluate_summary(summary, reference_summary)))
except (OSError, RuntimeError, ImportError) as error:
    summary = None
    print(f"Pegasus inference unavailable in this environment: {type(error).__name__}: {error}")


C:\Users\tuhin\Downloads\Text-Summarization-Models-main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-multi_news and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


C:\Users\tuhin\Downloads\Text-Summarization-Models-main\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tuhin\Downloads\Text-Summarization-Models-main\.model_cache\hub\models--google--pegasus-multi_news. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Summary: – Last year was the third warmest year on record, according to the latest report from the National Centers for Environmental Information. The agency says the average global temperature was 1.17 degrees Celsius, which was 1.34 degrees above the average of 1850-1900. The 10 warmest years in the record have all occurred since 2015, the New York Times reports. The Atlantic calls it "a stark reminder of the climate change we're facing today." The rest of the year was either the warmest or coolest on record. The ocean registered the highest amount of heat for the fifth year in a row, while land
Metrics: rouge1_f1=0.276, rouge2_f1=0.093, rougeL_f1=0.195, bleu4=0.062
